# 🛡️ HateGuard Multimodal 

**AI-Powered Toxicity Detection**
Supporting YouTube, Offline Media, and 100+ Languages (Hindi, English, etc.)

In [ ]:
import os
# 1. Cloning the repository (Shallow clone for 10x speed)
!rm -rf hate-comment-dectection
!git clone --depth 1 https://github.com/ATANU28-bit/hate-comment-dectection.git
%cd hate-comment-dectection

In [ ]:
# 2. Installing dependencies (Optimized for Colab speed)
print("Installing FFMPEG...")
!sudo apt install ffmpeg -y

print("Installing Python libraries...")
!pip install -r requirements.txt
!pip install --upgrade youtube-comment-downloader pytubefix uvicorn nest_asyncio yt-dlp

print("Installing UI dependencies (this can take 5 mins, please wait)...")
%cd ui
!npm install --quiet
%cd ..

### 3. One-Time YouTube Authentication
Run the cell below. It will show a link and a code. 
1. Click the link (google.com/device).
2. Enter the code shown in the output.
3. This authorizes audio downloads so you are not detected as a bot.

In [ ]:
from pytubefix import YouTube
import os
print("Starting Authentication flow...")
try:
    yt = YouTube('https://youtube.com/watch?v=jNQXAC9IVRw', use_oauth=True, allow_oauth_cache=True)
    _ = yt.streams.first() 
    print("\nSUCCESS: Login cached! You can now start the server.")
except Exception as e:
    print(f"Authentication error: {e}")

In [ ]:
import uvicorn
import nest_asyncio
import threading
import time
import sys
import subprocess
from src.api import app
from google.colab.output import eval_js

nest_asyncio.apply()

# 1. Start Backend
def run_backend():
    print("🚀 Starting Backend Server...")
    uvicorn.run(app, host="127.0.0.1", port=8000, log_level="info")

threading.Thread(target=run_backend, daemon=True).start()

# 2. Configure UI env
with open("ui/.env", "w", encoding="utf-8") as f:
    f.write("VITE_API_URL=/api\n")

# 3. Start Frontend
print("🎨 Starting Frontend UI...")
frontend = subprocess.Popen(
    ["npm", "run", "dev", "--prefix", "ui", "--", "--host", "127.0.0.1", "--port", "5173"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT
)

def stream_logs(proc):
    for line in iter(proc.stdout.readline, b''):
        sys.stdout.write(line.decode('utf-8'))
        sys.stdout.flush()

threading.Thread(target=stream_logs, args=(frontend,), daemon=True).start()

time.sleep(5)
proxy_url = eval_js("google.colab.kernel.proxyPort(5173)")
print(f"\n ✅ APP IS READY! OPEN LINK BELOW:\n {proxy_url}\n")

try:
    frontend.wait()
except KeyboardInterrupt:
    print("Shutting down...")
    frontend.terminate()